# Ejercicios resueltos: Introducción a la Regresión Lineal

Este notebook contiene la **resolución paso a paso** de los 4 ejercicios de la sección 9 de la *Práctica guiada de Regresión Lineal*.

>  **Consejo:** intentá resolver cada ejercicio por tu cuenta **antes** de mirar la solución. Después comparala con la tuya: hay muchas formas correctas de resolver un mismo ejercicio.

Cada ejercicio incluye el enunciado, el código comentado y la interpretación de los resultados.

## Preparación: importar las librerías

Importamos todo lo que vamos a necesitar en los cuatro ejercicios. Si te falta alguna librería, ejecutá antes `%pip install numpy pandas matplotlib seaborn scikit-learn`.

In [ ]:
import numpy as np                                    # cálculos numéricos
import pandas as pd                                   # tablas de datos
import matplotlib.pyplot as plt                       # gráficos
import seaborn as sns                                 # gráficos estadísticos

from sklearn.linear_model import LinearRegression     # el modelo
from sklearn.model_selection import train_test_split  # dividir en train / test
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score   # métricas

sns.set_theme(style="whitegrid")                      # estilo de los gráficos
print("¡Listo! Librerías importadas.")

### Función auxiliar para calcular las métricas

Como vamos a evaluar varios modelos, definimos una **función** que recibe los valores reales y los predichos y devuelve las 4 métricas (MAE, MSE, RMSE y R²) en un diccionario. Así evitamos repetir el mismo código en cada ejercicio.

In [ ]:
def calcular_metricas(y_verdadero, y_predicho):
    # Devuelve un diccionario con MAE, MSE, RMSE y R²
    mse = mean_squared_error(y_verdadero, y_predicho)
    return {
        "MAE": mean_absolute_error(y_verdadero, y_predicho),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R²": r2_score(y_verdadero, y_predicho),
    }

---
# Ejercicio 1: tu primer modelo desde cero

**Enunciado.** Generar 300 datos que sigan la relación $y = -5x + 10 + \varepsilon$, con $x$ entre 0 y 10 y ruido con desvío 2. Luego dividir en entrenamiento y prueba (80 % / 20 %), entrenar un modelo de regresión lineal, mostrar la pendiente y la ordenada aprendidas (¿se parecen a -5 y 10?) y calcular el RMSE y el R² en prueba.

## Paso 1: generar los datos

- `np.random.seed(10)`: fija la semilla para que los resultados sean siempre los mismos.
- `np.random.uniform(0, 10, n)`: genera `n` números aleatorios entre 0 y 10 (el atributo `X`).
- `np.random.randn(n) * 2`: genera ruido normal con media 0 y **desvío 2** (multiplicar por 2 amplía la dispersión).
- `X.reshape(-1, 1)`: convierte `X` en una tabla de 300 filas × 1 columna, que es el formato que pide Scikit-Learn.

In [ ]:
np.random.seed(10)

n = 300
X = np.random.uniform(0, 10, n)           # 300 valores entre 0 y 10
ruido = np.random.randn(n) * 2            # ruido con desvío 2
y = -5 * X + 10 + ruido                   # y = -5x + 10 + ruido

X_2d = X.reshape(-1, 1)                   # formato de tabla para Scikit-Learn

# Gráfico para mirar los datos antes de modelar
plt.figure(figsize=(10, 5))
plt.scatter(X, y, s=10, alpha=0.6)
plt.title("Ejercicio 1: datos generados con y = -5x + 10 + ruido")
plt.xlabel("X")
plt.ylabel("y")
plt.show()

Los puntos forman una nube que **baja de izquierda a derecha**: la relación es lineal y **negativa** (cuando `x` aumenta, `y` disminuye), lo cual es coherente con la pendiente -5.

## Paso 2: dividir en entrenamiento (80 %) y prueba (20 %)

Con `test_size=0.2` el 20 % de los datos queda para prueba y el 80 % restante para entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2d, y, test_size=0.2, random_state=42)

print("Entrenamiento:", X_train.shape[0], "registros")
print("Prueba:       ", X_test.shape[0], "registros")

## Paso 3: entrenar el modelo y mostrar lo que aprendió

Aplicamos el flujo de siempre: **instanciar → entrenar (`fit`) → predecir (`predict`)**, entrenando **solo con los datos de entrenamiento**.

In [ ]:
modelo = LinearRegression()               # 1. instanciar
modelo.fit(X_train, y_train)              # 2. entrenar

print(f"Pendiente aprendida:          {modelo.coef_[0]:.3f}   (real: -5)")
print(f"Ordenada al origen aprendida: {modelo.intercept_:.3f}   (real: 10)")
print(f"\nEcuación del modelo: y = {modelo.coef_[0]:.3f} * x + {modelo.intercept_:.3f}")

**Sí, se parecen mucho** a los valores reales (-5 y 10). No coinciden exactamente porque el ruido hace que el modelo nunca pueda recuperar los valores verdaderos a la perfección.

## Paso 4: predecir y evaluar en el conjunto de prueba

In [ ]:
y_test_pred = modelo.predict(X_test)      # 3. predecir sobre datos que el modelo NO vio

metricas = calcular_metricas(y_test, y_test_pred)
print(f"RMSE en prueba: {metricas['RMSE']:.3f}")
print(f"R² en prueba:   {metricas['R²']:.3f}")

**Interpretación:**

- **RMSE ≈ 1,7:** es cercano al desvío del ruido que agregamos (2). Es lo esperable: el ruido es aleatorio e **impredecible**, así que ningún modelo puede bajar sistemáticamente de ese error. (No da exactamente 2 porque con solo 60 datos de prueba la medición varía un poco de un sorteo a otro, como vimos en el ejercicio 2.) Un RMSE tan próximo al ruido indica que el modelo **captó prácticamente toda la información disponible**.
- **R² cercano a 1:** el modelo explica casi toda la variabilidad de `y`. Como `y` varía mucho (de aproximadamente -40 a 10) y el ruido es relativamente chico, el R² es muy alto.

## Paso 5 (extra): visualizar el resultado

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(X_train, y_train, s=10, alpha=0.5, label="Entrenamiento")
plt.scatter(X_test, y_test, s=10, alpha=0.8, color="orange", label="Prueba")
orden = np.argsort(X)                                  # para dibujar la recta de izquierda a derecha
plt.plot(X[orden], -5 * X[orden] + 10, "--", color="red", linewidth=2, label="Recta teórica")
plt.plot(X[orden], modelo.predict(X_2d[orden]), color="green", linewidth=2, label="Recta del modelo")
plt.title("Ejercicio 1: modelo vs. realidad")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()

---
# Ejercicio 2: ¿cuánto importa el tamaño del conjunto de prueba?

**Enunciado.** Con los datos de la sección 4 de la práctica ($y = 3x - 2 + \varepsilon$, 500 datos), probar `test_size` de **0.1**, **0.3** y **0.5**. Para cada uno, entrenar el modelo y anotar el RMSE y el R² en prueba. ¿Cambian mucho los resultados? ¿Por qué?

## Paso 1: volver a generar los datos de la sección 4

Los datos se generan igual que en la práctica: 500 valores de `X` entre -2 y 3, y `y = 3x - 2` más ruido de desvío 1.

In [ ]:
np.random.seed(42)

n = 500
X = np.linspace(-2, 3, n)                  # 500 valores equiespaciados entre -2 y 3
y = 3 * X - 2 + np.random.randn(n)         # y = 3x - 2 + ruido (desvío 1)
X_2d = X.reshape(-1, 1)

print("Datos generados:", X_2d.shape, y.shape)

## Paso 2: probar los tres tamaños de prueba con un bucle `for`

Un **bucle `for`** repite un bloque de código una vez por cada elemento de una lista. Acá la lista es `[0.1, 0.3, 0.5]`, y en cada vuelta la variable `tamaño` toma uno de esos valores.

Dentro del bucle repetimos los pasos de siempre (dividir, entrenar, predecir, evaluar) y guardamos los resultados en una lista de filas para armar una tabla al final.

In [ ]:
resultados = []                                        # lista vacía donde iremos guardando los resultados

for tamaño in [0.1, 0.3, 0.5]:
    # Dividimos con el tamaño de prueba de esta vuelta
    X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y, test_size=tamaño, random_state=42)

    # Instanciamos, entrenamos y predecimos
    modelo = LinearRegression().fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)

    # Calculamos las métricas sobre el conjunto de prueba y guardamos una fila
    m = calcular_metricas(y_te, y_pred)
    resultados.append({
        "test_size": tamaño,
        "Registros de prueba": len(y_te),
        "RMSE": m["RMSE"],
        "R²": m["R²"],
    })

pd.DataFrame(resultados).round(4)

## Paso 3: ¿por qué las diferencias son pequeñas pero existen?

En una **única** corrida, las diferencias entre los tres tamaños pueden parecer chicas, y al ser un solo sorteo aleatorio, es difícil saber cuánto se debe al tamaño y cuánto al azar de **qué datos cayeron en prueba**.

Para ver el efecto real, repetimos el experimento **30 veces** con distintos sorteos (cambiando `random_state`) y miramos el **promedio** y la **variabilidad** (desvío estándar) de cada métrica.

In [ ]:
filas = []

for tamaño in [0.1, 0.3, 0.5]:
    rmses, r2s = [], []                                 # listas para guardar los resultados de cada repetición
    for semilla in range(30):                           # 30 sorteos distintos
        X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y, test_size=tamaño, random_state=semilla)
        y_pred = LinearRegression().fit(X_tr, y_tr).predict(X_te)
        m = calcular_metricas(y_te, y_pred)
        rmses.append(m["RMSE"])
        r2s.append(m["R²"])

    filas.append({
        "test_size": tamaño,
        "RMSE promedio": np.mean(rmses),
        "RMSE desvío": np.std(rmses),                   # cuánto varía el RMSE entre sorteos
        "R² promedio": np.mean(r2s),
        "R² desvío": np.std(r2s),
    })

pd.DataFrame(filas).round(4)

### Conclusión del ejercicio 2

- **Los valores promedio casi no cambian** entre 0.1, 0.3 y 0.5: el modelo es muy simple (una recta con 2 parámetros) y 250 datos de entrenamiento alcanzan de sobra para aprenderlo bien.
- **Lo que sí cambia es la estabilidad** de la medición. Con `test_size=0.1` (solo 50 datos de prueba) las métricas **varían más de un sorteo a otro** (mayor desvío), porque con pocos datos de prueba el resultado depende mucho de cuáles tocaron.
- Con `test_size=0.5` la métrica es más estable, pero **queda menos data para entrenar** (250 de 500). En modelos más complejos o con pocos datos, esto sí perjudicaría el aprendizaje.

> **Regla práctica:** un conjunto de prueba **más grande** da una evaluación **más confiable**, y uno **más chico** deja más datos para entrenar. Por eso se suele usar entre 20 % y 30 % para prueba: es un buen equilibrio.

---
# Ejercicio 3: el efecto del ruido

**Enunciado.** Con la relación $y = 3x - 2 + \varepsilon$ (x entre -2 y 3, 500 datos), generar tres datasets con ruido de desvío **0.5**, **2** y **5**. Entrenar un modelo con cada uno y compararlos: ¿cómo cambian el R² y el RMSE al aumentar el ruido? ¿Se sigue aproximando el modelo a la pendiente real (3)? Graficar los tres casos.

## Paso 1: generar, dividir, entrenar y evaluar los tres datasets

Usamos un bucle `for` sobre los tres valores de ruido. **Importante:** para que la comparación sea justa, usamos **los mismos valores de `X`** y la **misma "semilla" de ruido** en los tres casos. Así lo único que cambia es la cantidad de ruido (el ruido base `ruido_base` se multiplica por el desvío deseado).

Guardamos también los datos y las predicciones de cada caso para graficarlos después.

In [ ]:
np.random.seed(0)

n = 500
X = np.linspace(-2, 3, n)
X_2d = X.reshape(-1, 1)
ruido_base = np.random.randn(n)             # ruido "unitario" (desvío 1), igual para los tres casos

resultados = []                             # para armar la tabla
guardado = {}                               # para guardar datos y modelos y graficar después

for desvio in [0.5, 2, 5]:
    y = 3 * X - 2 + ruido_base * desvio     # mismo X, el ruido se escala según el desvío

    X_tr, X_te, y_tr, y_te = train_test_split(X_2d, y, test_size=0.3, random_state=42)
    modelo = LinearRegression().fit(X_tr, y_tr)
    m = calcular_metricas(y_te, modelo.predict(X_te))

    resultados.append({
        "Desvío del ruido": desvio,
        "Pendiente aprendida": modelo.coef_[0],
        "Ordenada aprendida": modelo.intercept_,
        "RMSE (prueba)": m["RMSE"],
        "R² (prueba)": m["R²"],
    })
    guardado[desvio] = (y, modelo)

pd.DataFrame(resultados).round(3)

## Paso 2: graficar los tres casos

Dibujamos tres gráficos lado a lado (`plt.subplots(1, 3, ...)`) con **los mismos límites verticales** (`sharey=True`), para que la comparación visual sea justa. En cada uno mostramos los datos, la recta teórica y la recta del modelo.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for eje, desvio in zip(ejes, [0.5, 2, 5]):          # zip recorre eje y desvío "en paralelo"
    y, modelo = guardado[desvio]
    eje.scatter(X, y, s=6, alpha=0.5, label="Datos")
    eje.plot(X, 3 * X - 2, "--", color="red", linewidth=2, label="Recta teórica")
    eje.plot(X, modelo.predict(X_2d), color="green", linewidth=2, label="Recta del modelo")
    eje.set_title(f"Ruido con desvío {desvio}")
    eje.set_xlabel("X")

ejes[0].set_ylabel("y")
ejes[0].legend()
plt.tight_layout()
plt.show()

## Conclusiones del ejercicio 3

**1. ¿Cómo cambian el RMSE y el R² al aumentar el ruido?**

- El **RMSE sube**, y lo hace **casi igual al desvío del ruido** (≈ 0,5; ≈ 2; ≈ 5). Es lógico: el ruido es la parte impredecible, y marca un "piso" de error que ningún modelo puede superar.
- El **R² baja**: cuanto más ruido, menor es la proporción de la variabilidad de `y` que la recta puede explicar (el ruido "tapa" la señal).

**2. ¿Se sigue aproximando el modelo a la pendiente real (3)?**

**Sí, en los tres casos**, aunque con menos precisión cuando hay más ruido. El modelo es capaz de encontrar la tendencia general **incluso entre datos muy dispersos**, siempre que haya suficientes datos. Con ruido 5, la pendiente aprendida se aleja algo más de 3 que con ruido 0,5.

**3. Lo más importante:** el modelo hace "su mejor esfuerzo" en los tres casos (el RMSE ≈ ruido). Lo que cambia no es la calidad del modelo, sino **cuánta información hay en los datos para aprender**. Un R² bajo **no siempre indica un modelo malo**: a veces solo refleja datos muy ruidosos.

---
# Ejercicio 4 (desafío): predecir con un modelo múltiple

**Enunciado.** Agregar a la tabla `casas` (sección 6.6) una nueva columna `ambientes` con valores enteros entre 1 y 5. Modificar el precio para que **cada ambiente sume 5000 USD**. Reentrenar el modelo con **tres atributos** y verificar si el coeficiente de `ambientes` se parece a 5000.

## Paso 1: reconstruir la tabla de casas con la nueva columna

Repetimos la generación de la sección 6.6 y agregamos:

- `np.random.randint(1, 6, n_casas)`: genera números **enteros** aleatorios entre 1 y 5 (el 6 no se incluye).
- Una fórmula de precio actualizada, con el término `+ 5000 * ambientes`.

La "regla oculta" ahora es:

$$ precio = 1500 \cdot superficie - 800 \cdot antiguedad + 5000 \cdot ambientes + 20000 + \varepsilon $$

In [ ]:
np.random.seed(0)
n_casas = 300

casas = pd.DataFrame({
    "superficie_m2": np.random.uniform(30, 200, n_casas),     # entre 30 y 200 m²
    "antiguedad_anios": np.random.uniform(0, 50, n_casas),    # entre 0 y 50 años
    "ambientes": np.random.randint(1, 6, n_casas),            # enteros de 1 a 5
})

# Precio = regla oculta (ahora con ambientes) + ruido de desvío 10.000 USD
casas["precio_usd"] = (1500 * casas["superficie_m2"]
                       - 800 * casas["antiguedad_anios"]
                       + 5000 * casas["ambientes"]
                       + 20000
                       + np.random.normal(0, 10000, n_casas))

casas.head()

## Paso 2: separar atributos y etiqueta, dividir y entrenar

La única diferencia con el modelo de dos atributos es que **agregamos `"ambientes"` a la lista de columnas** de `X`. El resto del flujo es idéntico.

In [ ]:
atributos = ["superficie_m2", "antiguedad_anios", "ambientes"]   # ahora son 3 atributos
X_casas = casas[atributos]
y_casas = casas["precio_usd"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_casas, y_casas, test_size=0.25, random_state=42)

modelo_casas = LinearRegression().fit(Xc_train, yc_train)

## Paso 3: comparar los coeficientes aprendidos con los reales

Armamos una tabla que pone lado a lado el valor **real** (el que usamos para generar los datos) y el **aprendido** por el modelo.

In [ ]:
real = {"superficie_m2": 1500, "antiguedad_anios": -800, "ambientes": 5000}

comparacion = pd.DataFrame({
    "Coeficiente real": pd.Series(real),
    "Coeficiente aprendido": pd.Series(modelo_casas.coef_, index=atributos),
})
comparacion["Diferencia"] = comparacion["Coeficiente aprendido"] - comparacion["Coeficiente real"]

print(comparacion.round(1))
print(f"\nOrdenada al origen aprendida: {modelo_casas.intercept_:,.0f}   (real: 20.000)")

## Paso 4: evaluar el modelo

In [ ]:
m = calcular_metricas(yc_test, modelo_casas.predict(Xc_test))
print(f"RMSE en prueba: {m['RMSE']:,.0f} USD")
print(f"R² en prueba:   {m['R²']:.3f}")

## Conclusiones del ejercicio 4

- **Superficie y antigüedad** se recuperan con muy buena precisión.
- **El coeficiente de `ambientes` se parece a 5000, pero con menos exactitud.** Es el atributo con menos peso en el precio y varía solo entre 1 y 5, así que **el ruido (±10.000 USD) lo "tapa" más** que a los otros. Cuanto menor es el efecto de un atributo en comparación con el ruido, más difícil es medirlo con precisión.
- El **RMSE sigue siendo cercano a 10.000 USD**, que es justamente el ruido que pusimos. Es otra confirmación de que el modelo captó todo lo que se podía captar.

>  **Para probar:** aumentá `n_casas` (por ejemplo, a 3000) y volvé a ejecutar todo. Vas a ver que el coeficiente de `ambientes` se acerca mucho más a 5000: **con más datos, el modelo identifica mejor los efectos pequeños**.

## Paso 5 (extra): usar el modelo para predecir

¿Cuánto cuesta una casa de 80 m², 10 años de antigüedad y 3 ambientes? Le pasamos al modelo un DataFrame con **las mismas columnas** que usamos para entrenar.

In [ ]:
casa_nueva = pd.DataFrame({"superficie_m2": [80], "antiguedad_anios": [10], "ambientes": [3]})

precio_estimado = modelo_casas.predict(casa_nueva)[0]
precio_teorico = 1500 * 80 - 800 * 10 + 5000 * 3 + 20000      # sin ruido

print(f"Precio estimado por el modelo:      {precio_estimado:,.0f} USD")
print(f"Precio según la regla (sin ruido):  {precio_teorico:,.0f} USD")

---
#  Resumen de lo que practicaste

| Ejercicio | Qué aprendiste |
|---|---|
| **1** | El flujo completo: generar datos → dividir → entrenar → evaluar, con un modelo de pendiente negativa |
| **2** | El tamaño del conjunto de prueba afecta la **estabilidad** de la evaluación, no tanto el promedio |
| **3** | El ruido fija un **piso de error** (RMSE ≈ ruido) y baja el R², sin que el modelo sea "peor" |
| **4** | En regresión múltiple se puede comparar cada coeficiente con su valor real; los efectos pequeños son más difíciles de medir |

